In [ ]:
import polars as pl
import pyprojroot

ROOT = pyprojroot.here()

In [ ]:
data = pl.read_parquet(ROOT/"data"/"processed"/"temporada1_limpio.parquet")
data.columns

In [ ]:
features = [
    "relative_x", "relative_height", "distance_to_corner", "is_strike_zone", "is_shadow_zone",
    "release_speed", "pfx_x", "pfx_z", "strike_zone_size", "count",
    "stand", "p_throws", "pitcher", "pitch_type",
]

X = data.select(features)
y = data.select("swing")

In [ ]:
# ================================================================
# SWING PREDICTION — PIPELINE CORREGIDO
# LightGBM + Polars + Cross-Validation + Hyperparameter Tuning
# ================================================================

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
import matplotlib.pyplot as plt


# ----------------------------------------------------------------
# 1. CONVERSIÓN POLARS → PANDAS  [BUG FIX x2]
# ----------------------------------------------------------------
cat_cols = ["count", "stand", "p_throws", "pitcher", "pitch_type"]

X = X.with_columns([
    pl.col(c).cast(pl.String).cast(pl.Categorical) for c in cat_cols
])

# BUG FIX #1: sin pyarrow extension arrays — LightGBM requiere pandas estándar
X_pd = X.to_pandas()

# BUG FIX #2: target debe ser Series 1D, no DataFrame
y_pd = y["swing"].to_pandas()

# Cast a category de pandas (no Arrow)
for col in cat_cols:
    X_pd[col] = X_pd[col].astype('category')

print(f"X: {X_pd.shape} | Balance: {y_pd.mean():.3f} (swing rate)")

In [ ]:
borrar = data["pitch_type"].value_counts().sort("count")

In [ ]:
# ----------------------------------------------------------------
# 2. ESTRATEGIA DE CROSS-VALIDATION
# ----------------------------------------------------------------
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

In [ ]:
# ----------------------------------------------------------------
# 3. ESPACIO DE HIPERPARÁMETROS CORREGIDO
# ----------------------------------------------------------------
param_dist = {
    # Estructura del árbol
    'max_depth':          [5, 6, 7],
    'num_leaves':         [20, 31, 50],     # Siempre < 2^max_depth
    'min_child_samples':  [50, 100, 200],   # Mínimo de pitches por hoja

    # Gradiente y estocasticidad
    'learning_rate':      [0.01, 0.05, 0.1],
    'subsample':          [0.7, 0.8, 0.9],
    'colsample_bytree':   [0.7, 0.8, 1.0],

    # Regularización general
    'reg_alpha':          [0.0, 0.1, 0.5],  # L1
    'reg_lambda':         [1.0, 5.0, 10.0], # L2

    # Regularización para categóricas de alta cardinalidad (batter, pitcher)
    'cat_smooth':         [10, 30, 50],      # Prior bayesiano sobre IDs raros
    'min_data_per_group': [50, 100],         # Ignora IDs con muy pocas obs.
}

fixed_params = dict(
    objective='binary',
    n_estimators=200,    
    importance_type='gain',
    max_cat_to_onehot=4,     # Fuerza histogramas para batter/pitcher
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

In [ ]:
# ----------------------------------------------------------------
# 4. RANDOMIZED SEARCH CON CROSS-VALIDATION (LOGLOSS OPTIMIZADO)
# ----------------------------------------------------------------
search = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(**fixed_params), # Usando los params con n_estimators=150
    param_distributions=param_dist,
    n_iter=25,               # O los que hayas configurado
    cv=cv,
    scoring='neg_log_loss',  # <-- CAMBIO CLAVE: Scikit-learn usa logloss negativo
    refit=True,
    n_jobs=1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

# Entrenar
search.fit(X_pd, y_pd, categorical_feature=cat_cols)

# Imprimir resultados (le invertimos el signo para leer el LogLoss real)
print(f"\nMejor LogLoss (CV): {-search.best_score_:.4f}") 
print(f"Mejores params: {search.best_params_}")
best_params = search.best_params_

In [ ]:
# ----------------------------------------------------------------
# ANÁLISIS DE RESULTADOS DEL SEARCH
# ----------------------------------------------------------------
results = pd.DataFrame(search.cv_results_)

# Columnas clave — invertir signo para leer LogLoss real (positivo)
results['val_logloss']   = -results['mean_test_score']
results['train_logloss'] = -results['mean_train_score']
results['gap']           =  results['train_logloss'] - results['val_logloss']
# gap negativo  → val mejor que train  (raro, ruido)
# gap ~0        → bien generalizado
# gap positivo  → overfitting

# Ordenar por LogLoss de validación (menor es mejor)
results_sorted = (
    results
    .sort_values('val_logloss')
    [['rank_test_score', 'val_logloss', 'train_logloss', 'gap', 'params']]
    .reset_index(drop=True)
)

print("=== Top 10 combinaciones por LogLoss de validación ===")
print(results_sorted.head(10).to_string(index=False))

# Confirmar que best_params_ coincide con el rank 1
print(f"\nMejor LogLoss val : {results_sorted['val_logloss'].iloc[0]:.4f}")
print(f"Mejor LogLoss train: {results_sorted['train_logloss'].iloc[0]:.4f}")
print(f"Gap               : {results_sorted['gap'].iloc[0]:.4f}")
print(f"\nbest_params_: {search.best_params_}")

In [ ]:
# ----------------------------------------------------------------
# 3. CHEQUEO DE UNDERFITTING
#    Reentrenamos el modelo ganador UNA VEZ con eval_set
#    y miramos si el LogLoss seguía bajando en el árbol 150
#    o si ya se había aplanado (o empezaba a subir = overfit).
# ----------------------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_pd, y_pd, test_size=0.15, stratify=y_pd, random_state=42
)
 
check_params = dict(fixed_params)
check_params.update(best_params)
check_params['n_estimators'] = 500  # techo generoso solo para ver la curva completa
 
check_model = lgb.LGBMClassifier(**check_params)
check_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    categorical_feature=cat_cols,
)
 
eval_results = check_model.evals_result_['valid_0']['binary_logloss']
 
plt.figure(figsize=(8, 5))
plt.plot(eval_results)
plt.axvline(150, color='red', linestyle='--', label='n_estimators=150 (tu config actual)')
plt.axvline(np.argmin(eval_results), color='green', linestyle='--',
            label=f'mínimo real (árbol {np.argmin(eval_results)})')
plt.xlabel('Número de árboles')
plt.ylabel('LogLoss (validación)')
plt.title('¿150 árboles alcanza?')
plt.legend()
plt.tight_layout()
# plt.savefig(ROOT/"outputs"/"underfitting_check.png", dpi=150)
plt.show()
 
print(f"\nLogLoss en árbol 150:        {eval_results[149]:.4f}")
print(f"LogLoss mínimo (árbol {np.argmin(eval_results)}):  {min(eval_results):.4f}")
print(f"Diferencia: {eval_results[149] - min(eval_results):.4f}")
print("\nCómo leerlo:")
print("- Si la línea roja (150) está cerca del piso de la curva -> 150 está bien, no hay underfitting.")
print("- Si en el árbol 150 la curva sigue bajando claramente -> estás underfitteando, subí n_estimators.")
print("- Si el mínimo real está bien por encima de 150 (ej. árbol 300+) -> definitivamente subí el techo.")
 

In [ ]:
# ----------------------------------------------------------------
# 6. FEATURE IMPORTANCE (GAIN) + VISUALIZACIÓN
# ----------------------------------------------------------------
final_model = search.best_estimator_

importance_df = (
    pd.DataFrame({
        'feature':    X_pd.columns,
        'importance': final_model.feature_importances_
    })
    .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
ax.set_title('Feature Importance — Gain', fontsize=13)
ax.set_xlabel('Gain acumulado')
plt.tight_layout()
plt.show()